# Mt. Hood Infrastructure Exposure Assessment

Ties your Tephra2-simulated ash hazard (`max_ash_thickness.tif`, from `tephra2_grid_analysis.ipynb`) to real
infrastructure near Government Camp, Rhododendron, and Parkdale: roads, electric transmission lines and
structures, and drinking water and wastewater facilities. This is the same hazard-threshold GIS framework as
your Methods Report draft ("Tephra Hazard Exposure Assessment for Mt. Hood, Oregon"), with two differences:

- **Hazard source**: your own Tephra2 model output (kg/m² ash loading) instead of the published USGS hazard
  maps -- this is the piece that actually showcases your modeling work on the poster.
- **Thresholds**: the Wilson et al. (2014) / Jenkins et al. (2015) kg/m² thresholds already used throughout
  this project, instead of USGS's ≥1mm/≥10mm ashfall thresholds.

Everything else follows your Methods Report's described methodology directly: a buffer around each community,
reclassify the hazard layer into threshold polygons, intersect with infrastructure, and tabulate exposed
length (roads/lines) and facility counts (structures/drinking water/wastewater) per threshold per community.

### Data sources
All five infrastructure layers are loaded from manually downloaded local files -- public dataset per-county
splits and API access were both dead ends here (transmission structures and drinking water/wastewater
facilities don't have a clean live-API equivalent anyway), so this notebook just loads what you already have:

| Layer | Type | Files |
|---|---|---|
| Roads | lines | `roads_clackamas.shp`, `roads_hood_river.shp` |
| Transmission lines | lines | `BPA_TransmissionLines.shp` |
| Transmission structures | points | `BPA_TransmissionStructures.shp` |
| Drinking water facilities | points | `drinking water_clackamas_8_10_2026.csv`, `drinking water_hood_river_8_10_2026.csv` |
| Wastewater facilities | points | `wastewater_clackamas.json`, `wastewater_hood_river.json` |

If a filename doesn't match what you actually have, just edit the corresponding `LOCAL_*_PATH` variable in
Section 2 -- each accepts a single path or a list of paths (any mix of shapefile/GeoJSON/GeoPackage/CSV/JSON),
and multiple files are merged automatically.

### Hazard thresholds
| Threshold (kg/m²) | Impact |
|---:|---|
| 1 | Transport and agriculture disruption |
| 10 | Crop damage, infrastructure disruption |
| 100 | Roof collapse risk |
| 1,000 | Severe structural damage |

Wilson, T.M. et al. (2014); Jenkins, S.F. et al. (2015).

## Section 1 — Setup

Requires `max_ash_thickness.tif` (from `tephra2_grid_analysis.ipynb`'s Section 7) to already exist in the
working directory.

In [ ]:
import subprocess
subprocess.run('pip install --quiet geopandas', shell=True)

import os

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.features import shapes as raster_shapes
from shapely.geometry import Point, shape as shapely_shape
import matplotlib.pyplot as plt


In [ ]:
HAZARD_THRESHOLDS = [1, 10, 100, 1000]  # kg/m^2 -- Wilson et al. (2014), Jenkins et al. (2015)
UTM10N_EPSG = 32610   # WGS84 / UTM Zone 10N -- matches max_ash_thickness.tif and the rest of this project
WGS84_EPSG = 4326

BUFFER_RADIUS_M = 5_000  # 5 km, matching the Methods Report's community buffer

poi_names = ["Rhododendron", "Parkdale", "Govt. Camp"]
poi_locations = {
    "Rhododendron": (45.329563, -121.911191),
    "Parkdale": (45.519839, -121.596742),
    "Govt. Camp": (45.1808, -121.4509),
}

GEOTIFF_PATH = "max_ash_thickness.tif"


In [ ]:
poi_gdf = gpd.GeoDataFrame(
    {"name": list(poi_locations.keys())},
    geometry=[Point(lon, lat) for lat, lon in poi_locations.values()],
    crs=f"EPSG:{WGS84_EPSG}",
).to_crs(epsg=UTM10N_EPSG)

# 5 km buffer around each community centroid (matches the Methods Report's "4.2 Spatial Processing")
poi_buffers = poi_gdf.copy()
poi_buffers["geometry"] = poi_buffers.geometry.buffer(BUFFER_RADIUS_M)

print(f"Buffered {len(poi_buffers)} communities at {BUFFER_RADIUS_M/1000:.0f} km radius (CRS: EPSG:{UTM10N_EPSG})")
poi_buffers


## Section 2 — Infrastructure Data

`load_local_vector()` loads one or more manually downloaded files and merges them into a single GeoDataFrame
(reprojected to EPSG:32610). Each file can be:
- A real GIS format (shapefile/GeoJSON/GeoPackage) -- read via `geopandas.read_file()`.
- A CSV with latitude/longitude columns (matched case-insensitively against common names -- `latitude`,
  `lat`, `latitude83`, etc.) -- built into points directly, since `geopandas.read_file()` can't parse a plain
  CSV that way.
- A `.json` file that may or may not actually be valid GeoJSON -- tries `geopandas.read_file()` first, and if
  that fails, falls back to treating it as a flat JSON array of records with lat/lon columns (same as the CSV
  path).

If any file's lat/lon columns aren't recognized, the error message lists every column actually found in that
file, so you can add the real column name to `lat_col_candidates`/`lon_col_candidates` below.

In [ ]:
def _find_coord_column(columns, keywords):
    for col in columns:
        if col.strip().lower() in keywords:
            return col
    return None


LAT_KEYWORDS = {"latitude", "lat", "latitude83", "lat_dd83", "y_lat"}
LON_KEYWORDS = {"longitude", "long", "lon", "longitude83", "long_dd83", "x_lon"}


def _points_from_records_df(df, path, lat_keywords, lon_keywords):
    lat_col = _find_coord_column(df.columns, lat_keywords)
    lon_col = _find_coord_column(df.columns, lon_keywords)
    if lat_col is None or lon_col is None:
        raise ValueError(
            f"Couldn't find latitude/longitude columns in {path}. Found columns: {list(df.columns)}. "
            f"Add the real column name to LAT_KEYWORDS/LON_KEYWORDS above (lowercase) and try again."
        )
    has_coords = df[lat_col].notna() & df[lon_col].notna()
    n_missing = (~has_coords).sum()
    if n_missing:
        print(f"{path}: dropping {n_missing} of {len(df)} rows with no {lat_col}/{lon_col} value")
    df = df[has_coords]
    geometry = [Point(lon, lat) for lat, lon in zip(df[lat_col], df[lon_col])]
    return gpd.GeoDataFrame(df, geometry=geometry, crs=f"EPSG:{WGS84_EPSG}")


def _load_single_local_vector(path, lat_keywords=LAT_KEYWORDS, lon_keywords=LON_KEYWORDS):
    """Load one manually downloaded infrastructure file, in whatever format it turns out to be."""
    path_str = str(path).lower()
    if path_str.endswith(".csv"):
        gdf = _points_from_records_df(pd.read_csv(path), path, lat_keywords, lon_keywords)
    elif path_str.endswith(".json"):
        try:
            gdf = gpd.read_file(path)  # works if it's actually valid GeoJSON
            if gdf.crs is None:
                gdf = gdf.set_crs(epsg=WGS84_EPSG)
        except Exception:
            gdf = _points_from_records_df(pd.read_json(path), path, lat_keywords, lon_keywords)
    else:
        gdf = gpd.read_file(path)
        if gdf.crs is None:
            gdf = gdf.set_crs(epsg=WGS84_EPSG)
    return gdf.to_crs(epsg=UTM10N_EPSG)


def load_local_vector(paths, lat_keywords=LAT_KEYWORDS, lon_keywords=LON_KEYWORDS):
    """Load one or more manually downloaded infrastructure files and merge them into a single
    GeoDataFrame -- public datasets are often published per-county rather than as one combined file,
    so `paths` can be a single path or a list of paths (any mix of formats)."""
    if isinstance(paths, (str, os.PathLike)):
        paths = [paths]
    pieces = [_load_single_local_vector(p, lat_keywords, lon_keywords) for p in paths]
    merged = gpd.GeoDataFrame(pd.concat(pieces, ignore_index=True), crs=pieces[0].crs)
    print(f"Loaded {len(merged)} features from {len(pieces)} local file(s): {list(paths)}")
    return merged


### Roads (lines)

In [ ]:
LOCAL_ROADS_PATH = ["roads_clackamas.shp", "roads_hood_river.shp"]

roads_gdf = load_local_vector(LOCAL_ROADS_PATH)
roads_gdf.head()


### Electric transmission lines

In [ ]:
LOCAL_TRANSMISSION_LINES_PATH = "BPA_TransmissionLines.shp"

transmission_lines_gdf = load_local_vector(LOCAL_TRANSMISSION_LINES_PATH)
transmission_lines_gdf.head()


### Electric transmission structures (points)

Towers, poles, substations -- point features, so these get counted (like the facility layers below) rather
than measured as exposed length.

In [ ]:
LOCAL_TRANSMISSION_STRUCTURES_PATH = "BPA_TransmissionStructures.shp"

transmission_structures_gdf = load_local_vector(LOCAL_TRANSMISSION_STRUCTURES_PATH)
transmission_structures_gdf.head()


### Drinking water facilities (points)

In [ ]:
LOCAL_DRINKING_WATER_PATH = ["drinking water_clackamas_8_10_2026.csv", "drinking water_hood_river_8_10_2026.csv"]

drinking_water_gdf = load_local_vector(LOCAL_DRINKING_WATER_PATH)
drinking_water_gdf.head()


### Wastewater facilities (points)

In [ ]:
LOCAL_WASTEWATER_PATH = ["wastewater_clackamas.json", "wastewater_hood_river.json"]

wastewater_gdf = load_local_vector(LOCAL_WASTEWATER_PATH)
wastewater_gdf.head()


## Section 3 — Hazard Threshold Zones

Reclassifies `max_ash_thickness.tif` into a binary mask at each threshold and converts each mask into
dissolved AOI polygons -- the same "reclassify -> binary polygon -> dissolve" approach as the Methods
Report's "4.1 Hazard Threshold Framework", just applied to your Tephra2 GeoTIFF instead of a USGS raster.

In [ ]:
def build_threshold_polygons(geotiff_path, thresholds):
    """Return {threshold: dissolved shapely polygon (or None if the threshold isn't reached anywhere)}."""
    with rasterio.open(geotiff_path) as src:
        band = src.read(1)
        transform = src.transform
        raster_crs = src.crs
        nodata = src.nodata

    valid = band != nodata if nodata is not None else np.isfinite(band)

    polygons = {}
    for threshold in thresholds:
        mask = valid & (band >= threshold)
        if not mask.any():
            polygons[threshold] = None
            continue
        geoms = [shapely_shape(geom) for geom, value in raster_shapes(mask.astype("uint8"), mask=mask, transform=transform)
                 if value == 1]
        polygons[threshold] = gpd.GeoSeries(geoms, crs=raster_crs).union_all()
    return polygons, raster_crs


threshold_polygons, raster_crs = build_threshold_polygons(GEOTIFF_PATH, HAZARD_THRESHOLDS)
for threshold, poly in threshold_polygons.items():
    area_km2 = poly.area / 1e6 if poly is not None else 0.0
    print(f"{threshold:>5} kg/m^2 AOI: {area_km2:,.1f} km^2")


## Section 4 — Exposure Quantification

For each community's 5 km buffer: clip roads/transmission lines to the buffer and intersect with each
threshold's AOI polygon to get exposed length (km); for transmission structures, drinking water, and
wastewater facilities, count how many buffered points fall inside each threshold's AOI. Mirrors the Methods
Report's "4.3 Exposure Quantification" exactly, just with 2 line layers and 3 point layers instead of 1 each.

In [ ]:
def exposed_length_km(lines_gdf, buffer_geom, aoi_polygon):
    if aoi_polygon is None or lines_gdf.empty:
        return 0.0
    clipped = lines_gdf.clip(buffer_geom)
    if clipped.empty:
        return 0.0
    exposed = clipped.intersection(aoi_polygon)
    return exposed.length.sum() / 1000.0


def exposed_point_count(points_gdf, buffer_geom, aoi_polygon):
    if aoi_polygon is None or points_gdf.empty:
        return 0
    clipped = points_gdf.clip(buffer_geom)
    if clipped.empty:
        return 0
    return int(clipped.within(aoi_polygon).sum())


rows = []
for _, poi_row in poi_buffers.iterrows():
    name = poi_row["name"]
    buffer_geom = poi_row.geometry
    for threshold in HAZARD_THRESHOLDS:
        aoi_polygon = threshold_polygons[threshold]
        rows.append({
            "community": name,
            "threshold_kg_m2": threshold,
            "exposed_road_km": exposed_length_km(roads_gdf, buffer_geom, aoi_polygon),
            "exposed_transmission_line_km": exposed_length_km(transmission_lines_gdf, buffer_geom, aoi_polygon),
            "exposed_transmission_structure_count": exposed_point_count(transmission_structures_gdf, buffer_geom, aoi_polygon),
            "exposed_drinking_water_count": exposed_point_count(drinking_water_gdf, buffer_geom, aoi_polygon),
            "exposed_wastewater_count": exposed_point_count(wastewater_gdf, buffer_geom, aoi_polygon),
        })

exposure_summary = pd.DataFrame(rows)
exposure_summary.to_csv("infrastructure_exposure_summary.csv", index=False)
print("Wrote infrastructure_exposure_summary.csv")
exposure_summary


## Section 5 — Export for QGIS

Exports the exposed infrastructure (classified by the highest threshold each feature meets) and the threshold
AOI polygons themselves as GeoPackage layers -- load these alongside `max_ash_thickness.tif` in QGIS for the
poster figure, styled the same way as your Methods Report's cartographic design (hierarchical symbology,
hillshade base layer, semi-transparent AOI fills).

In [ ]:
def highest_threshold_met(geometry, thresholds, threshold_polygons, predicate):
    met = [t for t in thresholds if threshold_polygons[t] is not None and predicate(geometry, threshold_polygons[t])]
    return max(met) if met else None


def classify_by_threshold(gdf, thresholds, threshold_polygons, geom_predicate):
    if gdf.empty:
        gdf = gdf.copy()
        gdf["max_threshold_kg_m2"] = pd.Series(dtype="float")
        return gdf
    gdf = gdf.copy()
    gdf["max_threshold_kg_m2"] = gdf.geometry.apply(
        lambda geom: highest_threshold_met(geom, thresholds, threshold_polygons, geom_predicate)
    )
    return gdf[gdf["max_threshold_kg_m2"].notna()]


intersects_predicate = lambda geom, poly: geom.intersects(poly)
within_predicate = lambda geom, poly: geom.within(poly)

layers_to_export = {
    "exposed_roads": classify_by_threshold(roads_gdf, HAZARD_THRESHOLDS, threshold_polygons, intersects_predicate),
    "exposed_transmission_lines": classify_by_threshold(transmission_lines_gdf, HAZARD_THRESHOLDS, threshold_polygons, intersects_predicate),
    "exposed_transmission_structures": classify_by_threshold(transmission_structures_gdf, HAZARD_THRESHOLDS, threshold_polygons, within_predicate),
    "exposed_drinking_water": classify_by_threshold(drinking_water_gdf, HAZARD_THRESHOLDS, threshold_polygons, within_predicate),
    "exposed_wastewater": classify_by_threshold(wastewater_gdf, HAZARD_THRESHOLDS, threshold_polygons, within_predicate),
}

aoi_gdf = gpd.GeoDataFrame(
    {"threshold_kg_m2": [t for t, p in threshold_polygons.items() if p is not None]},
    geometry=[p for p in threshold_polygons.values() if p is not None],
    crs=raster_crs,
)

output_gpkg = "infrastructure_exposure.gpkg"
written_layers = []
for layer_name, gdf in layers_to_export.items():
    if len(gdf):
        gdf.to_file(output_gpkg, layer=layer_name, driver="GPKG")
        written_layers.append(layer_name)
aoi_gdf.to_file(output_gpkg, layer="hazard_threshold_aoi", driver="GPKG")
written_layers.append("hazard_threshold_aoi")

print(f"Wrote {output_gpkg} (CRS: EPSG:{UTM10N_EPSG}) with layers: {', '.join(written_layers)}")
